# U-Net Cityscapes Segmentation — Kaggle Training Notebook (19 Classes)

**Mục tiêu:**
- Huấn luyện kiến trúc U-Net tự xây dựng trên tập dữ liệu Cityscapes.
- Bài toán: Phân đoạn ngữ nghĩa (Semantic Segmentation).
- Số lớp: **19 lớp benchmark chuẩn** (road, person, car, v.v.), bỏ qua lớp void (ignore_index=255).
- Resolution: **512x256** (giảm từ 2048x1024 để fit RAM/VRAM Kaggle).

**Hướng dẫn sử dụng trên Kaggle:**
1. **Thêm dữ liệu:** Nhấn nút `+ Add Data` ở bên phải, tìm kiếm `cityscapes` (chọn bộ dữ liệu có cấu trúc `leftImg8bit` và `gtFine` hoặc cấu trúc ảnh/mask đơn giản).
2. **Thêm source code:** Có 2 cách:
   - Upload repo code lên làm Kaggle Dataset và Add Data.
   - Hoặc nén code thành file `.zip`, upload vào input, rồi unzip ra `/kaggle/working/`.
3. Chạy từng cell theo thứ tự.

**Cấu trúc dataset mong đợi (Code sẽ tự động nhận diện 1 trong 2 loại):**
- **Loại A (Cityscapes gốc):**
  ```
  /kaggle/input/cityscapes/leftImg8bit/train/city_name/...
  /kaggle/input/cityscapes/gtFine/train/city_name/...
  ```
- **Loại B (Đã làm phẳng - Flat):**
  ```
  /kaggle/input/cityscapes/train/images/...
  /kaggle/input/cityscapes/train/masks/...
  ```

## 1. Cài đặt môi trường

In [ ]:
# Kiểm tra thông tin GPU
!nvidia-smi

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install -q torch torchvision tqdm tensorboard Pillow opencv-python numpy matplotlib

In [ ]:
import os
import shutil

# --- CÀI ĐẶT SOURCE CODE TẠI ĐÂY ---
# Đổi REPO_SRC thành đường dẫn tới bộ mã nguồn của bạn trong /kaggle/input/
# Ví dụ: bạn upload repo dưới tên dataset là 'unet-cityscapes-code'
REPO_SRC = '/kaggle/input/unet-cityscapes-code'
WORK_DIR = '/kaggle/working/U-Net_Cityscapes_Segmentation'

if os.path.exists(REPO_SRC):
    print(f"Đang copy source code từ {REPO_SRC} sang {WORK_DIR}...")
    shutil.copytree(REPO_SRC, WORK_DIR, dirs_exist_ok=True)
    os.chdir(WORK_DIR)
    print(f'Thư mục làm việc hiện tại: {os.getcwd()}')
else:
    print(f"[CẢNH BÁO] Không tìm thấy source code tại {REPO_SRC}.")
    print("Vui lòng đảm bảo bạn đã Add Data chứa mã nguồn, hoặc upload file zip và giải nén thủ công!")
    # Tạo sẵn thư mục nếu bạn định kéo code từ github về
    # !git clone https://github.com/your_username/U-Net_Cityscapes_Segmentation.git
    # os.chdir('U-Net_Cityscapes_Segmentation')


In [ ]:
# DDP + AMP (giữ nguyên epoch/batch_size theo config mặc định)
%cd /kaggle/working/U-Net_Cityscapes_Segmentation
!torchrun --standalone --nproc_per_node=2 train.py \
  --data_root /kaggle/input/cityscapes \
  --amp

## 2. Import & Cấu hình Huấn luyện

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn

# Đảm bảo Python hiểu được các module nội bộ
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from models import UNet
from utils.dataloader import get_dataloader, CityscapesDataset, CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX
from utils.losses import CombinedLoss
from utils.metrics import SegmentationMetrics

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# =========================================================================
# CẤU HÌNH CÁC THAM SỐ HUẤN LUYỆN
# =========================================================================
CFG = {
    # Đường dẫn tới dataset Cityscapes trên Kaggle
    'data_root': '/kaggle/input/cityscapes', 
    
    # Cấu hình dữ liệu
    'img_size': (256, 512),       # (Height, Width)
    'label_type': 'labelIds',     # 'labelIds' (0-33) cần convert sang 'trainIds' (0-18). 
                                  # Nếu dataset của bạn đã có file `*trainIds.png`, hãy để 'trainIds'.

    # Kiến trúc Model
    'base_channels': 64,
    'bilinear': True,

    # Tham số huấn luyện
    'epochs': 60,                 
    'batch_size': 8,              # Tăng lên 16 nếu GPU P100 (16GB VRAM) chịu được
    'num_workers': 2,             # Kaggle thường chỉ cho phép 2 workers tối ưu
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'alpha_loss': 0.5,            # Tỉ lệ kết hợp: 0.5 * WeightedCE + 0.5 * DiceLoss

    # Lưu trữ
    'save_dir': '/kaggle/working/checkpoints',
    'resume': None,               # Nếu muốn tiếp tục, để path tới checkpoint. VD: '/kaggle/working/checkpoints/last.pth'
    
    # Cài đặt chung
    'seed': 42,
    'log_interval': 50,
}

os.makedirs(CFG['save_dir'], exist_ok=True)

# Cố định random seed
import random
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

## 3. Trực quan hóa Dữ liệu (Visualization)

In [ ]:
# Khởi tạo Dataset để lấy mẫu
try:
    sample_dataset = CityscapesDataset(
        root=CFG['data_root'], 
        split='train',
        img_size=CFG['img_size'], 
        augment=False,
        label_type=CFG['label_type']
    )
    print(f"Đã load Dataset mode: {sample_dataset.mode}")
except Exception as e:
    print(f"LỖI KHỞI TẠO DATASET: {e}")
    print("Hãy kiểm tra lại đường dẫn `CFG['data_root']` đã trỏ đúng vào dataset Cityscapes chưa.")

In [ ]:
# Vẽ thử vài ảnh
from utils.dataloader import decode_mask

MEAN = np.array([0.485, 0.456, 0.406])[:, None, None]
STD  = np.array([0.229, 0.224, 0.225])[:, None, None]

if 'sample_dataset' in locals() and len(sample_dataset) > 0:
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    fig.suptitle('Mẫu dữ liệu Cityscapes (19 classes)', fontsize=14, fontweight='bold')

    for i in range(3):
        # Chọn mẫu ngẫu nhiên
        idx = random.randint(0, len(sample_dataset) - 1)
        img_t, mask_t = sample_dataset[idx]

        # Phục hồi ảnh RGB gốc
        img_np = (img_t.numpy() * STD + MEAN).clip(0, 1).transpose(1, 2, 0)
        
        # Ánh xạ nhãn sang màu
        mask_color = decode_mask(mask_t)

        axes[i, 0].imshow(img_np);     axes[i, 0].set_title('Input RGB'); axes[i, 0].axis('off')
        axes[i, 1].imshow(mask_color); axes[i, 1].set_title('Mask 19-class'); axes[i, 1].axis('off')
        
        # Biểu đồ Histogram các lớp xuất hiện trong mask này
        mask_flat = mask_t.numpy().flatten()
        mask_flat = mask_flat[mask_flat != IGNORE_INDEX] # Bỏ qua lớp void 255
        
        bins = np.arange(NUM_CLASSES + 1) - 0.5
        axes[i, 2].hist(mask_flat, bins=bins, color='steelblue', rwidth=0.8)
        axes[i, 2].set_xticks(range(NUM_CLASSES))
        axes[i, 2].set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
        axes[i, 2].set_title('Phân phối Lớp')

    plt.tight_layout()
    plt.show()

## 4. Chuẩn bị DataLoader, Model, Loss, Optimizer

In [ ]:
print("[INFO] Tạo DataLoaders...")
train_loader = get_dataloader(
    root=CFG['data_root'], split='train',
    batch_size=CFG['batch_size'], img_size=CFG['img_size'],
    num_workers=CFG['num_workers'], augment=True,
    label_type=CFG['label_type']
)

val_loader = get_dataloader(
    root=CFG['data_root'], split='val',
    batch_size=CFG['batch_size'], img_size=CFG['img_size'],
    num_workers=CFG['num_workers'], augment=False,
    label_type=CFG['label_type']
)

In [ ]:
print("[INFO] Khởi tạo U-Net Model...")
model = UNet(
    in_channels=3,
    num_classes=NUM_CLASSES,
    bilinear=CFG['bilinear'],
    base_channels=CFG['base_channels'],
).to(device)

# Khởi tạo Loss (CombinedLoss = Weighted CE + Dice)
loss_fn = CombinedLoss(
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    alpha=CFG['alpha_loss']
)

# Optimizer AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG['lr'],
    weight_decay=CFG['weight_decay']
)

# Scheduler: Cosine Annealing
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=CFG['epochs'], 
    eta_min=CFG['lr'] / 100
)

start_epoch = 1
best_miou = 0.0

if CFG['resume'] and os.path.exists(CFG['resume']):
    print(f"[INFO] Đang load checkpoint từ {CFG['resume']}")
    ckpt = torch.load(CFG['resume'], map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optim_state'])
    start_epoch = ckpt['epoch'] + 1
    best_miou = ckpt['best_miou']
    print(f"Tiếp tục từ epoch {start_epoch}, best_mIoU = {best_miou:.4f}")

## 5. Vòng lặp Huấn luyện (Training Loop)

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_miou': [], 'val_acc': []}

print(f"{'='*60}")
print(f"Bắt đầu huấn luyện {CFG['epochs']} epoch")
print(f"{'='*60}")

for epoch in range(start_epoch, CFG['epochs'] + 1):
    t0 = time.time()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\n── Epoch {epoch}/{CFG['epochs']}  (lr={current_lr:.6f}) ──")

    # ========================== TRAIN ==========================
    model.train()
    train_loss, train_ce, train_dice = 0.0, 0.0, 0.0
    
    for i, (imgs, masks) in enumerate(train_loader, 1):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(imgs)
        loss, ce, dice = loss_fn(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_loss += loss.item()
        train_ce += ce.item()
        train_dice += dice.item()

        if i % CFG['log_interval'] == 0 or i == len(train_loader):
            print(f"  [Train] Batch {i:4d}/{len(train_loader)} | Loss: {loss.item():.4f} | CE: {ce.item():.4f} | Dice: {dice.item():.4f}")

    train_loss /= len(train_loader)

    # ========================== VALID ==========================
    model.eval()
    val_loss = 0.0
    metrics = SegmentationMetrics(NUM_CLASSES, IGNORE_INDEX, CLASS_NAMES)

    with torch.no_grad():
        for i, (imgs, masks) in enumerate(val_loader, 1):
            imgs = imgs.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(imgs)
            loss, _, _ = loss_fn(logits, masks)
            val_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            metrics.update(preds, masks)
            
    val_loss /= len(val_loader)
    res = metrics.compute()
    miou = res['mIoU']
    pacc = res['pixel_accuracy']

    # Bước scheduler
    scheduler.step()

    # Lưu lịch sử
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_miou'].append(miou)
    history['val_acc'].append(pacc)

    # Tổng kết epoch
    elapsed = time.time() - t0
    print(f"\n  >> Tóm tắt Epoch {epoch}:")
    print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"  Val mIoU:   {miou:.4f} | Pixel Acc: {pacc:.4f}")
    print(f"  Thời gian:  {elapsed:.1f}s")

    # Lưu Model
    is_best = miou > best_miou
    if is_best:
        best_miou = miou
        print(f"  ★ Đạt mIoU tốt nhất mới: {best_miou:.4f}")

    ckpt_state = {
        'epoch': epoch,
        'best_miou': best_miou,
        'model_state': model.state_dict(),
        'optim_state': optimizer.state_dict(),
        'config': CFG,
    }
    
    torch.save(ckpt_state, f"{CFG['save_dir']}/last.pth")
    if is_best:
        torch.save(ckpt_state, f"{CFG['save_dir']}/best_model.pth")

print(f"\nHoàn tất huấn luyện. Best mIoU: {best_miou:.4f}")

## 6. Biểu đồ quá trình học

In [ ]:
epochs_arr = range(1, len(history['train_loss']) + 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(epochs_arr, history['train_loss'], label='Train Loss')
ax[0].plot(epochs_arr, history['val_loss'], label='Val Loss')
ax[0].set_title('Loss Curve')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss')
ax[0].legend(); ax[0].grid(True, linestyle='--', alpha=0.6)

ax[1].plot(epochs_arr, history['val_miou'], label='Val mIoU', color='green')
ax[1].plot(epochs_arr, history['val_acc'], label='Val Pixel Acc', color='orange', linestyle='--')
ax[1].set_title('Metrics')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Score')
ax[1].legend(); ax[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 7. Đánh giá trên tập Test (hoặc suy luận)

In [ ]:
# Load best model
best_path = f"{CFG['save_dir']}/best_model.pth"
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    print("Đã tải model tốt nhất.")
else:
    print("Không tìm thấy best_model.pth")

In [ ]:
# Thử suy luận 1 vài ảnh ngẫu nhiên từ tập val (do tập test Cityscapes gốc không có nhãn public)
if 'sample_dataset' in locals():
    val_dataset = CityscapesDataset(
        root=CFG['data_root'], 
        split='val', 
        img_size=CFG['img_size'], 
        augment=False,
        label_type=CFG['label_type']
    )
    
    fig, axes = plt.subplots(4, 3, figsize=(15, 16))
    fig.suptitle('Kết quả Suy luận (Val Set)', fontsize=14)
    
    with torch.no_grad():
        for i in range(4):
            idx = random.randint(0, len(val_dataset)-1)
            img_t, mask_t = val_dataset[idx]
            
            # Chuẩn bị input
            inp = img_t.unsqueeze(0).to(device)
            out = model(inp)
            pred = torch.argmax(out, dim=1).squeeze(0).cpu()
            
            # Đảo Normalize ảnh
            img_np = (img_t.numpy() * STD + MEAN).clip(0,1).transpose(1,2,0)
            
            # Decode masks
            gt_col = decode_mask(mask_t)
            pr_col = decode_mask(pred)
            
            axes[i,0].imshow(img_np); axes[i,0].set_title('RGB'); axes[i,0].axis('off')
            axes[i,1].imshow(gt_col); axes[i,1].set_title('Ground Truth'); axes[i,1].axis('off')
            axes[i,2].imshow(pr_col); axes[i,2].set_title('Prediction'); axes[i,2].axis('off')
            
    plt.tight_layout()
    plt.show()

In [ ]:
import os

ckpt_candidates = [
    "/kaggle/working/checkpoints/best_model.pth",
    "/kaggle/working/U-Net_Cityscapes_Segmentation/checkpoints/best_model.pth",
]
ckpt_path = next((p for p in ckpt_candidates if os.path.exists(p)), None)
if not ckpt_path:
    raise FileNotFoundError("best_model.pth not found. Train first or update the path.")

!python submit.py --checkpoint {ckpt_path} --data_root /kaggle/input/cityscapes --output_dir /kaggle/working/submission